# Compute DSWRF Solar Flux Index

Extracts the downward shortwave radiation flux (DSWRF) at the grid cell nearest to **25°N, 280°E**
from NCEP/NCAR Reanalysis-2 daily Gaussian-grid files, resamples to Sunday-ending weekly means,
and appends `dswrf` as a new column to each reef-site CSV file.

**Data source:** NCEP/NCAR Reanalysis-2 — `dswrf.sfc.gauss.YYYY.nc`  
Download: <https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis2/Dailies/gaussian_grid/>

**Historical data (1985–2024):** Copied from the pre-computed reference file  
`bleaching_predict_clean_repo/Scripts/previous_scripts/dswrf_sfc_weekly.csv`.  
Values verified to match the resampled NetCDF values to floating-point precision for all weeks
except 2024 week 52 (9.7 W/m² difference — likely a NCEP2 data revision for that most-recent week;
the old value is kept for consistency with model training data).

**New data (2025–present):** Resampled directly from the downloaded NetCDF files.

**Re-running:** Add the new year's NetCDF to `NC_DIR`, update `NC_GLOB`, and re-run.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path

print(f'numpy   {np.__version__}')
print(f'pandas  {pd.__version__}')
print(f'xarray  {xr.__version__}')

## Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR   = Path('../Data')
SITE_FILES = {
    'sombrero': DATA_DIR / 'sombrero_preds.csv',
    'sk':       DATA_DIR / 'sk_preds.csv',
    'molasses': DATA_DIR / 'molasses_preds.csv',
}

# Historical reference: pre-computed dswrf from the old processing pipeline
OLD_DSWRF = Path(
    '/Users/marcodia/Documents/Projects/bleaching_predict_clean_repo'
    '/Scripts/previous_scripts/dswrf_sfc_weekly.csv'
)

# New NetCDF files (NCEP/NCAR Reanalysis-2, daily Gaussian grid)
# Add new year files here when available
NC_DIR  = Path('/Users/marcodia/Downloads')
NC_GLOB = 'dswrf.sfc.gauss.202*.nc'

# Grid cell for the Loop Current / Caribbean study area
LAT_TARGET = 25.0   # degrees N
LON_TARGET = 280.0  # degrees E (NCEP uses 0–360 convention)

print('Configuration loaded.')
print(f'Grid cell target: {LAT_TARGET}N, {LON_TARGET}E')

## Load Historical dswrf Reference (1985–2024)

Reconstitutes the `(iso_year, iso_week)` mapping for each row of the old CSV using the same
Sunday-ending weekly resampling that generated it.

In [ ]:
old = pd.read_csv(OLD_DSWRF)
old['dswrf'] = pd.to_numeric(old['dswrf'], errors='coerce')
print(f'Old reference rows: {len(old)}, NaN: {old["dswrf"].isna().sum()}')

# Reconstruct (iso_year, iso_week) by simulating the original add_week_53() sequence.
# The old file covers W-SUN dates from 1985-01-01 to 2024-12-31.
dates_old = pd.date_range('1985-01-01', '2024-12-31', freq='W-SUN')
iso_cal   = pd.DatetimeIndex(dates_old).isocalendar()
iso_years = iso_cal.year.values.astype(int)
iso_weeks = iso_cal.week.values.astype(int)

reco_iso_years, reco_iso_weeks, reco_is_nan = [], [], []
for i in range(len(dates_old)):
    reco_iso_years.append(iso_years[i])
    reco_iso_weeks.append(iso_weeks[i])
    reco_is_nan.append(False)
    # Insert phantom week-53 NaN where the year transitions 52→1 without a real week 53
    if i < len(dates_old) - 1 and iso_weeks[i] == 52 and iso_weeks[i+1] == 1:
        reco_iso_years.append(iso_years[i])
        reco_iso_weeks.append(53)
        reco_is_nan.append(True)

assert len(reco_iso_years) == len(old), \
    f'Reconstruction mismatch: {len(reco_iso_years)} vs {len(old)}'

df_old = pd.DataFrame({
    'iso_year':   reco_iso_years,
    'iso_week':   reco_iso_weeks,
    'is_phantom': reco_is_nan,
    'dswrf':      old['dswrf'].values,
})
print(f'Reconstructed: {len(df_old)} rows, '
      f'{df_old["dswrf"].isna().sum()} NaN (phantom week-53)')
print(f'Coverage: {reco_iso_years[0]} wk{reco_iso_weeks[0]} → '
      f'{reco_iso_years[-1]} wk{reco_iso_weeks[-1]}')

## Load and Resample New NetCDF Files (2024–present)

Extracts the nearest grid cell and resamples to Sunday-ending weekly means.

In [ ]:
nc_files = sorted(NC_DIR.glob(NC_GLOB))
print(f'Found {len(nc_files)} NetCDF files:')
for f in nc_files:
    print(f'  {f.name}')

ds_new = xr.concat([xr.open_dataset(f) for f in nc_files], dim='time')
print(f'\nCombined time range: {str(ds_new.time.values[0])[:10]} '
      f'to {str(ds_new.time.values[-1])[:10]}')

pt = ds_new.dswrf.sel(lat=LAT_TARGET, lon=LON_TARGET, method='nearest')
print(f'Grid cell used: lat={float(pt.lat):.4f}N, lon={float(pt.lon):.4f}E')

weekly_new  = pt.resample(time='W-SUN').mean()
iso_new     = pd.DatetimeIndex(weekly_new.time.values).isocalendar()
df_new = pd.DataFrame({
    'iso_year': iso_new.year.values.astype(int),
    'iso_week': iso_new.week.values.astype(int),
    'dswrf':    weekly_new.values.astype(float),
    'date':     [str(t)[:10] for t in weekly_new.time.values],
})
print(f'Weekly resampled: {len(df_new)} weeks, '
      f'{df_new.iloc[0].iso_year} wk{df_new.iloc[0].iso_week} → '
      f'{df_new.iloc[-1].iso_year} wk{df_new.iloc[-1].iso_week}')

## Verify 2024 Values

Compares 2024 weekly values between the reference file and the newly resampled NetCDF.

In [ ]:
old_2024 = df_old[(df_old['iso_year'] == 2024) & (~df_old['is_phantom'])]
new_2024 = df_new[df_new['iso_year'] == 2024]
m2024 = old_2024.merge(new_2024.rename(columns={'dswrf':'dswrf_new'}),
                        on=['iso_year','iso_week'], how='inner')
diff = (m2024['dswrf'] - m2024['dswrf_new']).abs()

print(f'2024 matched weeks: {len(m2024)}')
print(f'Max absolute difference: {diff.max():.4f} W/m²')
print(f'Rows with diff > 0.001 W/m²: {(diff > 0.001).sum()}')
print()

if diff.max() > 1.0:
    print('NOTE: differences > 1 W/m² are due to NCEP2 data revisions for recent weeks.')
    print('The old reference value is kept for consistency with model training data.\n')
    discrepant = m2024[diff > 1.0]
    print('Discrepant weeks:')
    print(discrepant[['iso_year','iso_week','dswrf','dswrf_new']].to_string(index=False))

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(m2024['iso_week'], m2024['dswrf'], 'o-', ms=4, label='Old reference', color='steelblue')
ax.plot(m2024['iso_week'], m2024['dswrf_new'], 's--', ms=4, label='New NetCDF', color='tomato', alpha=0.7)
ax.set_xlabel('ISO week (2024)', fontsize=12)
ax.set_ylabel('dswrf (W/m²)', fontsize=12)
ax.set_title('2024 Verification: Old Reference vs. New NetCDF', fontsize=12)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Build Combined Lookup and Update CSVs

Uses old reference values for 1985–2024 and new NetCDF values for 2025+.

In [ ]:
# Historical: 1985–2024 real observations (exclude phantom week-53 rows)
df_hist = df_old[~df_old['is_phantom']][['iso_year','iso_week','dswrf']].copy()

# New: 2025 onwards
df_ext  = df_new[df_new['iso_year'] >= 2025][['iso_year','iso_week','dswrf']].copy()

df_combined = pd.concat([df_hist, df_ext], ignore_index=True)
print(f'Combined lookup: {len(df_combined)} rows')
print(f'  iso_year range: {df_combined.iso_year.min()} – {df_combined.iso_year.max()}')

for site, csv_path in SITE_FILES.items():
    df = pd.read_csv(csv_path)
    n_orig = len(df)
    if 'dswrf' in df.columns:
        df = df.drop(columns=['dswrf'])

    df = df.merge(
        df_combined[['iso_year','iso_week','dswrf']],
        left_on=['year','week'],
        right_on=['iso_year','iso_week'],
        how='left',
    ).drop(columns=['iso_year','iso_week'])

    assert len(df) == n_orig, f'Row count mismatch for {site}!'
    n_filled = df['dswrf'].notna().sum()
    n_nan    = df['dswrf'].isna().sum()
    df.to_csv(csv_path, index=False)
    print(f'{site}: {n_filled} filled, {n_nan} NaN → {csv_path.name}')

print('\nAll CSV files updated.')

## Final Verification

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

for ax, (site, csv_path) in zip(axes, SITE_FILES.items()):
    df = pd.read_csv(csv_path)
    df_plot = df.dropna(subset=['dswrf'])
    ax.plot(df_plot['totalWeek'], df_plot['dswrf'], linewidth=0.7, color='darkorange')
    ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
    ax.set_ylabel('dswrf (W/m²)', fontsize=10)
    ax.set_title(site.capitalize(), fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.4)
    n_nan = df['dswrf'].isna().sum()
    print(f'{site}: range=[{df["dswrf"].min():.1f}, {df["dswrf"].max():.1f}] W/m², NaN={n_nan}')

axes[-1].set_xlabel('totalWeek (sequential)', fontsize=11)
plt.suptitle('DSWRF Solar Flux Added to CSV Files', fontsize=13)
plt.tight_layout()
plt.show()

# Spot-check 2024/2025 boundary
df_s = pd.read_csv(SITE_FILES['sombrero'])
m = ((df_s['year'] == 2024) & (df_s['week'] >= 50)) | \
    ((df_s['year'] == 2025) & (df_s['week'] <= 3))
print('\n2024/2025 boundary rows:')
print(df_s.loc[m, ['year','week','totalWeek','dswrf']].to_string(index=False))

print('\nLast non-NaN rows:')
last_valid = df_s[df_s['dswrf'].notna()].tail(3)
print(df_s.loc[last_valid.index, ['year','week','totalWeek','dswrf']].to_string(index=False))